In [ ]:
import os
import random
import warnings
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc, accuracy_score, precision_score,
    recall_score, f1_score
)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════════════════════╗
# ║  1. CONFIGURATION                                          ║
# ╚══════════════════════════════════════════════════════════════╝
class CFG:
    DATA_ROOT       = '/kaggle/input/breast-histopathology-images'
    OUT_DIR         = '/kaggle/working'
    MAGNIFICATIONS  = ['40X', '100X', '200X', '400X']
    IMG_SIZE        = 224
    BATCH_SIZE      = 32
    # Loss
    FOCAL_GAMMA     = 2.0
    FOCAL_ALPHA     = 0.33       # benign upweighting (benign ≈ 33% of dataset)
    LABEL_SMOOTH    = 0.05
    # Seeds — pool of seeds to draw from. Each mag picks the first N_SEEDS
    # from this list (N_SEEDS is now per-mag, see MAG_CFG).
    SEEDS           = [42, 123, 2024, 7, 1337]
    # General
    WEIGHT_DECAY    = 5e-3
    DROPOUT_2       = 0.3
    # SWA — see v0-10 fix #2
    SWA_FRAC        = 0.5        # SWA window = SWA_FRAC * unfreeze_epochs.
                                 # Raised from 0.25 in v0-9 (window was too late).
    SWA_LR_FACTOR   = 1.0        # SWA LR = lr_backbone_p2 * factor
    # Test-time augmentation
    TTA_FIVE_CROP   = True       # add 5-crop (center + 4 corners) views
    DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    AMP_ENABLED     = True       # mixed precision

# Per-magnification hyperparameters — v0-10
# Inherits v0-9's MAG_CFG for 40X/100X/200X verbatim (they performed well).
# 400X changes (v0-10 fix #3):
#   - PATIENCE 25 → 15  (v0-9 ran 30+ epochs of patience burn after best ep)
#   - AUG_STRENGTH 'strong' triggers heavier ColorJitter + RandomErasing
# All mags now declare N_SEEDS individually (v0-10 fix #4):
#   - 40X / 100X / 200X: 3 seeds
#   - 400X: 5 seeds  (lift was largest there, more affordable per seed)
MAG_CFG = {
    '40X':  dict(FREEZE_EPOCHS=15, UNFREEZE_EPOCHS=80,  DROPOUT_1=0.5, PATIENCE=35,
                 LR_P1=3e-4, LR_HEAD_P2=1e-4, LR_BACKBONE_P2=1e-5,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3,
                 IMG_SIZE=224, AUG_STRENGTH='normal', N_SEEDS=3),
    '100X': dict(FREEZE_EPOCHS=30, UNFREEZE_EPOCHS=90,  DROPOUT_1=0.5, PATIENCE=35,
                 LR_P1=3e-4, LR_HEAD_P2=1e-4, LR_BACKBONE_P2=1e-5,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3,
                 IMG_SIZE=224, AUG_STRENGTH='normal', N_SEEDS=3),
    '200X': dict(FREEZE_EPOCHS=12, UNFREEZE_EPOCHS=80,  DROPOUT_1=0.6, PATIENCE=35,
                 LR_P1=2e-4, LR_HEAD_P2=8e-5, LR_BACKBONE_P2=8e-6,
                 WEIGHT_DECAY=5e-3, P2_WARMUP=3,
                 IMG_SIZE=224, AUG_STRENGTH='normal', N_SEEDS=3),
    '400X': dict(FREEZE_EPOCHS=25, UNFREEZE_EPOCHS=100, DROPOUT_1=0.5, PATIENCE=15,   # ← was 40 then 25
                 LR_P1=3e-4, LR_HEAD_P2=8e-5, LR_BACKBONE_P2=8e-6,
                 WEIGHT_DECAY=6e-3, P2_WARMUP=3,
                 IMG_SIZE=320, AUG_STRENGTH='strong',  N_SEEDS=5),                    # ← stronger aug, 5 seeds
}

def set_seed(seed):
    """Seed everything reproducibly while ALLOWING cudnn.benchmark for speed."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    # NOTE: cudnn.deterministic intentionally NOT set to True.
    # The previous setting cost ~20% throughput for no real determinism
    # (DataLoader workers and augmentations broke bit-exact reproducibility
    # anyway). Seed-averaging in N_SEEDS makes results stable.

torch.backends.cudnn.benchmark = True   # auto-tunes best conv kernels

os.makedirs(CFG.OUT_DIR, exist_ok=True)
print(f"Device         : {CFG.DEVICE}")
print(f"Magnifications : {CFG.MAGNIFICATIONS}")
print(f"Seed pool      : {CFG.SEEDS}")
print(f"Seeds per mag  : " + ", ".join(f"{m}:{MAG_CFG[m]['N_SEEDS']}" for m in CFG.MAGNIFICATIONS))
print(f"AMP            : {CFG.AMP_ENABLED}")
print(f"cudnn.benchmark: {torch.backends.cudnn.benchmark}\n")

# ╔══════════════════════════════════════════════════════════════╗
# ║  2. LOSS                                                   ║
# ╚══════════════════════════════════════════════════════════════╝
def focal_loss(logits, labels):
    """Focal BCE with alpha balancing.
    alpha < 0.5 upweights the positive class — here malignants are
    over-represented (~67%) so we set alpha = 0.33 to give benigns
    more weight per-sample (factor 1-alpha for malignant, alpha for benign).
    """
    smooth_labels = labels * (1 - CFG.LABEL_SMOOTH) + 0.5 * CFG.LABEL_SMOOTH
    bce = F.binary_cross_entropy_with_logits(logits, smooth_labels, reduction='none')
    pt  = torch.where(labels == 1, torch.sigmoid(logits), 1 - torch.sigmoid(logits))
    # alpha_t: lower weight on majority (malignant=1), higher on minority (benign=0)
    alpha_t = torch.where(labels == 1,
                          torch.full_like(labels, CFG.FOCAL_ALPHA),
                          torch.full_like(labels, 1.0 - CFG.FOCAL_ALPHA))
    return (alpha_t * (1 - pt) ** CFG.FOCAL_GAMMA * bce).mean()

# ╔══════════════════════════════════════════════════════════════╗
# ║  3. MODEL (ResNet50 + MS-HRA)                              ║
# ╚══════════════════════════════════════════════════════════════╝
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class MS_HRA_Block(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1    = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1      = nn.BatchNorm2d(out_ch)
        self.conv2    = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2      = nn.BatchNorm2d(out_ch)
        self.ca       = ChannelAttention(out_ch)
        self.sa       = SpatialAttention()
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out * self.ca(out)
        out = out * self.sa(out)
        return F.relu(out + self.shortcut(x))

class Res_MSHRA(nn.Module):
    def __init__(self, dropout_1=0.5):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.layer0 = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.mshra  = nn.Sequential(
            MS_HRA_Block(1024, 1024, stride=1),
            MS_HRA_Block(1024, 2048, stride=2))
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(), nn.BatchNorm1d(2048), nn.Dropout(dropout_1),
            nn.Linear(2048, 512), nn.ReLU(inplace=True),
            nn.BatchNorm1d(512), nn.Dropout(CFG.DROPOUT_2), nn.Linear(512, 1))
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)

    def backbone_params(self):
        return (list(self.layer0.parameters()) + list(self.layer1.parameters()) +
                list(self.layer2.parameters()) + list(self.layer3.parameters()))
    def new_params(self):
        return list(self.mshra.parameters()) + list(self.head.parameters())
    def freeze_backbone(self):
        for p in self.backbone_params(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone_params(): p.requires_grad = True
    def forward(self, x):
        x = self.layer0(x); x = self.layer1(x)
        x = self.layer2(x); x = self.layer3(x)
        x = self.mshra(x);  x = self.gap(x)
        return self.head(x)

class TemperatureScaler(nn.Module):
    """Multi-step temperature scaling via LBFGS.

    v0-9 ran a single opt.step(nll) here and got the SAME T (≈2.58) for
    every (mag, seed) — LBFGS's internal max_iter wasn't enough on its
    own to move raw_T far from the init point. Restored to a loop of
    short LBFGS calls (max_iter=20 each), with early break on
    convergence. Typically converges in 3-5 outer iters.

    The v0-7/v0-8 50-iter loop was overkill but at least was correct.
    """
    def __init__(self, base_model):
        super().__init__()
        self.model = base_model
        self.raw_T = nn.Parameter(torch.zeros(1))
    @property
    def temperature(self):
        return F.softplus(self.raw_T) + 0.5
    def forward(self, x):
        return self.model(x) / self.temperature
    def calibrate(self, loader, max_outer=15, tol=1e-4):
        self.model.eval()
        opt = torch.optim.LBFGS([self.raw_T], lr=0.05, max_iter=20)
        L, Y = [], []
        with torch.no_grad():
            for imgs, labels, _, _ in loader:
                if CFG.AMP_ENABLED:
                    with torch.cuda.amp.autocast():
                        logits = self.model(imgs.to(CFG.DEVICE)).squeeze(1)
                    L.append(logits.float().cpu())
                else:
                    L.append(self.model(imgs.to(CFG.DEVICE)).squeeze(1).cpu())
                Y.append(labels)
        L = torch.cat(L); Y = torch.cat(Y)
        if roc_auc_score(Y.numpy(), torch.sigmoid(L).numpy()) < 0.5: L = -L
        def nll():
            opt.zero_grad()
            loss = F.binary_cross_entropy_with_logits(
                L / (F.softplus(self.raw_T.cpu()) + 0.5), Y)
            loss.backward(); return loss
        prev_T = float('inf')
        for _ in range(max_outer):
            opt.step(nll)
            cur_T = self.temperature.item()
            if abs(cur_T - prev_T) < tol: break
            prev_T = cur_T
        return self.temperature.item()

# ╔══════════════════════════════════════════════════════════════╗
# ║  4. DATA                                                   ║
# ╚══════════════════════════════════════════════════════════════╝
imagenet_norm = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

def get_train_transform(img_size, strength='normal'):
    """Training augmentations.

    strength='normal': v0-9 default — works well for 40X/100X/200X.
    strength='strong': v0-10 addition for 400X. Heavier ColorJitter and
        RandomErasing to combat fast overfitting at 400X (seeds in v0-9
        all hit best AUC at P2 ep 1-3, then 30+ epochs of patience burn).
        Stain/color variation is the dominant source of variance at high
        magnification, so we attack it directly.
    """
    if strength == 'strong':
        cj_b, cj_c, cj_s, cj_h = 0.4, 0.4, 0.3, 0.1
        gray_p   = 0.1
        erase_p  = 0.25
        erase_sc = (0.02, 0.15)
    else:
        cj_b, cj_c, cj_s, cj_h = 0.2, 0.2, 0.15, 0.05
        gray_p   = 0.05
        erase_p  = 0.15
        erase_sc = (0.02, 0.08)
    return transforms.Compose([
        transforms.Resize((int(img_size*1.15), int(img_size*1.15))),
        transforms.RandomCrop(img_size),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=cj_b, contrast=cj_c,
                               saturation=cj_s, hue=cj_h),
        transforms.RandomGrayscale(p=gray_p),
        transforms.ToTensor(), imagenet_norm,
        transforms.RandomErasing(p=erase_p, scale=erase_sc)])

def get_val_transform(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(), imagenet_norm])

class HistologyDataset(Dataset):
    def __init__(self, mag, split, transform=None):
        self.samples   = []
        self.transform = transform
        # Auto-discover the actual data root (handles nested Kaggle zip paths)
        data_root = CFG.DATA_ROOT
        for root, dirs, _ in os.walk(data_root):
            if mag in dirs:
                data_root = root
                break
        for cls, lbl in [('benign', 0), ('malignant', 1)]:
            path = os.path.join(data_root, mag, split, cls)
            if os.path.exists(path):
                for f in sorted(os.listdir(path)):
                    if f.lower().endswith(('.png','.jpg','.jpeg')):
                        self.samples.append((os.path.join(path, f), lbl))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, lbl = self.samples[idx]
        img    = Image.open(path).convert('RGB')
        tensor = self.transform(img) if self.transform else transforms.ToTensor()(img)
        return tensor, torch.tensor(lbl, dtype=torch.float32), img, path
    def get_labels(self):
        return [lbl for _, lbl in self.samples]

def _collate(b):
    return (torch.stack([x[0] for x in b]),
            torch.stack([x[1] for x in b]),
            [x[2] for x in b], [x[3] for x in b])

def make_train_loader(ds):
    """Class-balanced sampling: each sample weighted by 1/freq(class).
    Combined with focal-alpha, this addresses BreaKHis's 2:1 imbalance
    that hurt specificity in v0-2 through v0-7 (e.g., 40X v0-3 had
    Sens=0.93 but Spec=0.84).
    """
    labels        = np.array(ds.get_labels())
    class_counts  = np.bincount(labels.astype(int), minlength=2)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels.astype(int)]
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(ds), replacement=True)
    # drop_last=True keeps BN stats stable (tiny final batches were polluting BN running stats)
    # Note: sampler is mutually exclusive with shuffle=True, so we omit shuffle.
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                      num_workers=4, pin_memory=True, drop_last=True,
                      collate_fn=_collate)

def make_val_loader(ds):
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                      num_workers=4, pin_memory=True, drop_last=False,
                      collate_fn=_collate)

# ╔══════════════════════════════════════════════════════════════╗
# ║  5. TRAINING + EVAL HELPERS                                ║
# ╚══════════════════════════════════════════════════════════════╝
def train_one_epoch(model, loader, optimizer, scaler):
    """AMP-enabled training step.
    autocast wraps forward+loss; backward/step go through GradScaler.
    Grad-clip applied AFTER unscale_ to use real (not scaled) grad norms.
    """
    model.train()
    total = 0.0; n = 0
    for imgs, labels, _, _ in loader:
        imgs   = imgs.to(CFG.DEVICE, non_blocking=True)
        labels = labels.to(CFG.DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if CFG.AMP_ENABLED:
            with torch.cuda.amp.autocast():
                logits = model(imgs).squeeze(1)
                loss   = focal_loss(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs).squeeze(1)
            loss   = focal_loss(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total += loss.item() * imgs.size(0); n += imgs.size(0)
    return total / max(n, 1)

@torch.no_grad()
def evaluate_auc(model, loader):
    model.eval()
    probs, targets = [], []
    for imgs, labels, _, _ in loader:
        if CFG.AMP_ENABLED:
            with torch.cuda.amp.autocast():
                logits = model(imgs.to(CFG.DEVICE, non_blocking=True)).squeeze(1)
        else:
            logits = model(imgs.to(CFG.DEVICE, non_blocking=True)).squeeze(1)
        probs.extend(torch.sigmoid(logits.float()).cpu().numpy())
        targets.extend(labels.numpy())
    return roc_auc_score(targets, probs)

@torch.no_grad()
def tta_predict(m, loader, img_size):
    """11-view TTA: 6 rot/flip + (optionally) 5 crops.
    Five-crop adds center + 4 corners at 90% of input size resized back up.
    Stacks geometrically with rotation/flip views.
    """
    m.eval()
    all_probs, all_labels, all_imgs = [], [], []
    crop_size = int(img_size * 0.9)
    for imgs, labels, orig_imgs, _ in loader:
        imgs = imgs.to(CFG.DEVICE, non_blocking=True)
        # 6-view base (rotations + flips)
        views = [imgs,
                 torch.flip(imgs, [3]),
                 torch.flip(imgs, [2]),
                 torch.flip(imgs, [2, 3]),
                 torch.rot90(imgs, 1, [2, 3]),
                 torch.rot90(imgs, 3, [2, 3])]
        # Optional: 5 additional spatial crops (center + corners)
        if CFG.TTA_FIVE_CROP:
            H, W = imgs.shape[2], imgs.shape[3]
            ch, cw = crop_size, crop_size
            # (top, left) pairs: center, top-left, top-right, bottom-left, bottom-right
            offsets = [((H-ch)//2, (W-cw)//2),
                       (0, 0), (0, W-cw),
                       (H-ch, 0), (H-ch, W-cw)]
            for (top, left) in offsets:
                crop = imgs[:, :, top:top+ch, left:left+cw]
                crop = F.interpolate(crop, size=(H, W), mode='bilinear', align_corners=False)
                views.append(crop)
        if CFG.AMP_ENABLED:
            with torch.cuda.amp.autocast():
                preds = sum(torch.sigmoid(m(v).float()) for v in views) / len(views)
        else:
            preds = sum(torch.sigmoid(m(v)) for v in views) / len(views)
        all_probs.extend(preds.squeeze(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_imgs.extend(orig_imgs)
    return np.array(all_probs), np.array(all_labels), all_imgs

def denorm(t):
    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    return (t.cpu()*std+mean).clamp(0,1).permute(1,2,0).numpy()

def accuracy_optimal_threshold(y_true, y_prob):
    """NOTE: This tunes the threshold on the same split we report on.
    That's test-set leakage. User is fixing it in a follow-up — leaving here
    for now to keep numbers comparable to v0-2..v0-7.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    accs = [accuracy_score(y_true, (y_prob >= t).astype(int)) for t in thresholds]
    return thresholds[np.argmax(accs)], max(accs)

# ╔══════════════════════════════════════════════════════════════╗
# ║  6. XAI HELPERS  (unchanged from v0-7)                     ║
# ╚══════════════════════════════════════════════════════════════╝
class GradCAM:
    def __init__(self, model, target_layer, img_size):
        self.model = model
        self.img_size = img_size
        self.gradients = None; self.activations = None
        target_layer.register_forward_hook(
            lambda m,i,o: setattr(self, 'activations', o.detach()))
        target_layer.register_full_backward_hook(
            lambda m,gi,go: setattr(self, 'gradients', go[0].detach()))
    def __call__(self, img_tensor):
        self.model.eval()
        logit = self.model(img_tensor)
        self.model.zero_grad()
        logit.squeeze().backward()
        weights = self.gradients.mean(dim=[2,3], keepdim=True)
        cam     = F.relu((weights * self.activations).sum(1, keepdim=True))
        cam     = F.interpolate(cam, (self.img_size, self.img_size),
                                mode='bilinear', align_corners=False)
        cam     = cam.squeeze().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

def collect_examples(val_ds, y_prob, n=4):
    benign, malignant = [], []
    for i in range(len(val_ds)):
        img_t, lbl, orig, path = val_ds[i]
        prob = float(y_prob[i])
        if int(lbl.item()) == 0 and len(benign) < n:
            benign.append((img_t, orig, prob))
        elif int(lbl.item()) == 1 and len(malignant) < n:
            malignant.append((img_t, orig, prob))
        if len(benign) >= n and len(malignant) >= n: break
    return benign, malignant

def plot_gradcam_for_mag(model, val_ds, y_prob, mag, img_size, n=4):
    cam_engine = GradCAM(model, model.mshra[1].conv2, img_size)
    benign, malignant = collect_examples(val_ds, y_prob, n)
    fig, axes = plt.subplots(4, n, figsize=(4*n, 16))
    for col, (img_t, orig, prob) in enumerate(benign):
        orig_np = denorm(img_t)
        with torch.enable_grad():
            hmap = cam_engine(img_t.unsqueeze(0).to(CFG.DEVICE))
        overlay = (0.55*orig_np + 0.45*plt.cm.jet(hmap)[:,:,:3]).clip(0,1)
        axes[0][col].imshow(orig_np)
        axes[0][col].set_title(f"Benign  p={prob:.3f}", fontsize=10)
        axes[0][col].axis('off')
        axes[1][col].imshow(overlay)
        axes[1][col].set_title("Grad-CAM", fontsize=10)
        axes[1][col].axis('off')
    for col, (img_t, orig, prob) in enumerate(malignant):
        orig_np = denorm(img_t)
        with torch.enable_grad():
            hmap = cam_engine(img_t.unsqueeze(0).to(CFG.DEVICE))
        overlay = (0.55*orig_np + 0.45*plt.cm.jet(hmap)[:,:,:3]).clip(0,1)
        axes[2][col].imshow(orig_np)
        axes[2][col].set_title(f"Malignant  p={prob:.3f}", fontsize=10)
        axes[2][col].axis('off')
        axes[3][col].imshow(overlay)
        axes[3][col].set_title("Grad-CAM", fontsize=10)
        axes[3][col].axis('off')
    for row, label in enumerate(['Benign — original','Benign — Grad-CAM',
                                  'Malignant — original','Malignant — Grad-CAM']):
        axes[row][0].set_ylabel(label, fontsize=10, rotation=90, labelpad=12)
    fig.suptitle(f"Grad-CAM  |  {mag}  |  Target: mshra[1].conv2\n"
                 "Warm=high activation  |  p=P(malignant)", fontsize=12)
    plt.tight_layout(); plt.show()

def plot_attention_for_mag(model, val_ds, y_prob, mag, img_size, n=6):
    stored = {'b1': None, 'b2': None}
    hooks  = [
        model.mshra[0].sa.register_forward_hook(
            lambda m,i,o: stored.update({'b1': o.detach().cpu()})),
        model.mshra[1].sa.register_forward_hook(
            lambda m,i,o: stored.update({'b2': o.detach().cpu()})),
    ]
    benign, malignant = collect_examples(val_ds, y_prob, n//2)
    samples     = benign + malignant
    labels_list = ['Benign']*(n//2) + ['Malignant']*(n//2)
    fig, axes   = plt.subplots(n, 4, figsize=(14, 3.5*n))
    for row, ((img_t, orig, prob), cls_name) in enumerate(zip(samples, labels_list)):
        with torch.no_grad():
            model(img_t.unsqueeze(0).to(CFG.DEVICE))
        orig_np = denorm(img_t)
        b1 = F.interpolate(stored['b1'], (img_size, img_size),
                           mode='bilinear', align_corners=False).squeeze().numpy()
        b2 = F.interpolate(stored['b2'], (img_size, img_size),
                           mode='bilinear', align_corners=False).squeeze().numpy()
        overlay = (0.55*orig_np + 0.45*plt.cm.hot(b2)[:,:,:3]).clip(0,1)
        for col, (img_show, cmap) in enumerate([(orig_np,None),(b1,'hot'),(b2,'hot'),(overlay,None)]):
            axes[row][col].imshow(img_show, cmap=cmap, **(dict(vmin=0,vmax=1) if cmap else {}))
            axes[row][col].axis('off')
        if row == 0:
            for col, title in enumerate(['Original','Block-1 attn','Block-2 attn','Block-2 overlay']):
                axes[0][col].set_title(title, fontsize=9)
        axes[row][0].set_ylabel(f"{cls_name}\np={prob:.3f}", fontsize=9)
    for h in hooks: h.remove()
    fig.suptitle(f"MS-HRA Spatial Attention  |  {mag}  |  Bright=high weight", fontsize=12)
    plt.tight_layout(); plt.show()

def plot_prediction_grid_for_mag(val_ds, y_prob, y_pred, mag, n_each=4):
    cats = {'TN':[],'TP':[],'FP':[],'FN':[]}
    for i in range(len(val_ds)):
        img_t, lbl, _, _ = val_ds[i]
        tl=int(lbl.item()); pl=int(y_pred[i]); p=float(y_prob[i])
        if   tl==0 and pl==0: cats['TN'].append((img_t,p))
        elif tl==1 and pl==1: cats['TP'].append((img_t,p))
        elif tl==0 and pl==1: cats['FP'].append((img_t,p))
        elif tl==1 and pl==0: cats['FN'].append((img_t,p))
    cats['TN'].sort(key=lambda x: x[1])
    cats['TP'].sort(key=lambda x: -x[1])
    cats['FP'].sort(key=lambda x: -x[1])
    cats['FN'].sort(key=lambda x: x[1])
    row_meta = [
        ('True Negatives (correct benign)',    'TN','#2ecc71'),
        ('True Positives (correct malignant)', 'TP','#2ecc71'),
        ('False Positives (benign→malignant)', 'FP','#e74c3c'),
        ('False Negatives (malignant→benign)', 'FN','#e74c3c'),
    ]
    fig, axes = plt.subplots(4, n_each, figsize=(4*n_each, 16))
    for row, (row_label, key, border_col) in enumerate(row_meta):
        for col in range(n_each):
            ax = axes[row][col]
            if col < len(cats[key]):
                img_t, prob = cats[key][col]
                ax.imshow(denorm(img_t))
                ax.set_title(f"p={prob:.3f}", fontsize=9)
                for spine in ax.spines.values():
                    spine.set_edgecolor(border_col); spine.set_linewidth(4)
            else:
                ax.text(0.5,0.5,'N/A',ha='center',va='center',
                        transform=ax.transAxes,color='gray')
            ax.set_xticks([]); ax.set_yticks([])
        axes[row][0].set_ylabel(row_label, fontsize=9, rotation=90, labelpad=12)
    green_p = mpatches.Patch(color='#2ecc71', label='Correct')
    red_p   = mpatches.Patch(color='#e74c3c', label='Incorrect')
    fig.legend(handles=[green_p,red_p], loc='lower center', ncol=2, fontsize=10,
               bbox_to_anchor=(0.5,-0.01))
    fig.suptitle(f"Sample Prediction Grid  |  {mag}\n"
                 "Green=correct  Red=error  p=P(malignant)", fontsize=12)
    plt.tight_layout(); plt.show()

def plot_calibration_for_mag(y_true, y_prob, mag, n_bins=10):
    edges  = np.linspace(0,1,n_bins+1)
    b_acc=[]; b_conf=[]; b_size=[]
    for i in range(n_bins):
        mask = (y_prob>=edges[i]) & (y_prob<edges[i+1])
        if mask.sum()>0:
            b_acc.append(y_true[mask].mean())
            b_conf.append(y_prob[mask].mean())
            b_size.append(mask.sum())
        else:
            b_acc.append(np.nan); b_conf.append((edges[i]+edges[i+1])/2); b_size.append(0)
    b_acc=np.array(b_acc); b_conf=np.array(b_conf); b_size=np.array(b_size)
    valid = ~np.isnan(b_acc)
    ece   = (b_size[valid]/len(y_true)*np.abs(b_acc[valid]-b_conf[valid])).sum()
    fig, axes = plt.subplots(1,2,figsize=(12,4))
    axes[0].plot([0,1],[0,1],'--',color='gray',lw=1.5,label='Perfect')
    axes[0].bar(b_conf[valid], b_acc[valid], width=0.08, alpha=0.75,
                color='steelblue', edgecolor='white', label='Model')
    axes[0].bar(b_conf[valid], b_conf[valid], width=0.08, alpha=0.25,
                color='tomato', edgecolor='none', label='Gap')
    axes[0].set_xlim(0,1); axes[0].set_ylim(0,1)
    axes[0].set_title(f'Reliability diagram  |  ECE={ece:.4f}')
    axes[0].set_xlabel('Mean predicted probability'); axes[0].set_ylabel('Fraction positive')
    axes[0].legend(fontsize=9)
    qual = 'Excellent' if ece<0.03 else 'Good' if ece<0.05 else 'Moderate' if ece<0.10 else 'Poor'
    axes[0].text(0.04,0.92,f"ECE={ece:.4f}\n{qual}",transform=axes[0].transAxes,fontsize=10,
                 bbox=dict(boxstyle='round',facecolor='wheat',alpha=0.6))
    axes[1].hist(y_prob[y_true==0],bins=25,alpha=0.65,label='Benign',color='steelblue')
    axes[1].hist(y_prob[y_true==1],bins=25,alpha=0.65,label='Malignant',color='tomato')
    axes[1].set_title('Confidence distribution'); axes[1].legend()
    plt.suptitle(f"Calibration Analysis  |  {mag}", fontsize=12)
    plt.tight_layout(); plt.show()
    return ece

# ╔══════════════════════════════════════════════════════════════╗
# ║  7. TRAIN ONE SEED ON ONE MAGNIFICATION                    ║
# ╚══════════════════════════════════════════════════════════════╝
def train_one_seed(mag, seed, mc, train_loader, val_loader):
    """Trains a single (mag, seed) combination with two-phase + SWA.
    Returns (calibrated_model, swa_model_or_none, final_temperature).
    """
    set_seed(seed)
    freeze_epochs   = mc['FREEZE_EPOCHS']
    unfreeze_epochs = mc['UNFREEZE_EPOCHS']
    dropout_1       = mc['DROPOUT_1']
    patience        = mc['PATIENCE']
    lr_p1           = mc['LR_P1']
    lr_head_p2      = mc['LR_HEAD_P2']
    lr_backbone_p2  = mc['LR_BACKBONE_P2']
    weight_decay    = mc['WEIGHT_DECAY']
    p2_warmup       = mc['P2_WARMUP']

    model    = Res_MSHRA(dropout_1=dropout_1).to(CFG.DEVICE)
    scaler   = torch.cuda.amp.GradScaler(enabled=CFG.AMP_ENABLED)
    best_auc = 0.0
    history  = {'loss': [], 'val_auc': [], 'phase': []}
    ckpt     = os.path.join(CFG.OUT_DIR, f'res_mshra_{mag}_s{seed}.pth')

    # Phase 1 — backbone frozen
    print(f"\n  [seed {seed}] Phase 1 — backbone frozen ({freeze_epochs} epochs)")
    model.freeze_backbone()
    opt1   = torch.optim.AdamW(model.new_params(), lr=lr_p1, weight_decay=weight_decay)
    sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=freeze_epochs, eta_min=1e-6)
    for epoch in range(1, freeze_epochs + 1):
        loss    = train_one_epoch(model, train_loader, opt1, scaler)
        val_auc = evaluate_auc(model, val_loader)
        sched1.step()
        history['loss'].append(loss); history['val_auc'].append(val_auc); history['phase'].append(1)
        is_best = val_auc > best_auc
        if is_best: best_auc = val_auc; torch.save(model.state_dict(), ckpt)
        print(f"    P1 {epoch:02d}/{freeze_epochs} | LR:{opt1.param_groups[0]['lr']:.5f} | "
              f"Loss:{loss:.4f} | AUC:{val_auc:.4f}" + (" ←" if is_best else ""))

    # ═════════════════════════════════════════════════════════════════
    # Phase 2 — full fine-tuning + SWA
    # ═════════════════════════════════════════════════════════════════
    # v0-10 SWA logic (fix #2):
    #   - swa_target_start: ideal epoch to enter SWA (governed by SWA_FRAC).
    #   - swa_budget: how many epochs SWA runs once activated.
    #   - Force-activation: if patience would exhaust BEFORE swa_target_start,
    #     we activate SWA at that moment instead of early-stopping. This
    #     guarantees SWA always gets at least swa_budget epochs to explore,
    #     which it didn't get in v0-9 (only 3/12 activations, all w/ 1-2 ep).
    #   - During SWA: ckpt selection FREEZES (best pre-SWA weights are kept
    #     untouched on disk) and patience is DISABLED. SWA gets exactly its
    #     full budget regardless of validation behavior.
    swa_target_start = max(int(unfreeze_epochs * (1 - CFG.SWA_FRAC)), p2_warmup + 1)
    swa_budget       = max(int(unfreeze_epochs * CFG.SWA_FRAC), 5)   # floor 5 ep
    print(f"\n  [seed {seed}] Phase 2 — full fine-tuning (up to {unfreeze_epochs} ep)")
    print(f"  [seed {seed}]   SWA target_start=ep{swa_target_start}, budget={swa_budget} ep, "
          f"force-on-patience=ON")
    model.unfreeze_backbone()
    patience_ctr = 0
    opt2 = torch.optim.AdamW([
        {'params': model.backbone_params(), 'lr': lr_backbone_p2},
        {'params': model.new_params(),      'lr': lr_head_p2},
    ], weight_decay=weight_decay)
    p2_cosine_epochs = unfreeze_epochs - p2_warmup
    sched2 = torch.optim.lr_scheduler.SequentialLR(opt2, schedulers=[
        torch.optim.lr_scheduler.LinearLR(
            opt2, start_factor=0.1, end_factor=1.0, total_iters=p2_warmup),
        torch.optim.lr_scheduler.CosineAnnealingLR(
            opt2, T_max=p2_cosine_epochs, eta_min=1e-7),
    ], milestones=[p2_warmup])

    swa_model = AveragedModel(model)
    swa_lr    = lr_backbone_p2 * CFG.SWA_LR_FACTOR
    swa_sched = SWALR(opt2, swa_lr=swa_lr, anneal_epochs=2, anneal_strategy='linear')
    swa_active           = False
    swa_activation_epoch = None
    pre_swa_best_auc     = 0.0      # locks in best ckpt AUC at SWA activation

    actual_p2_epochs = 0
    for epoch in range(1, unfreeze_epochs + 1):
        loss    = train_one_epoch(model, train_loader, opt2, scaler)
        val_auc = evaluate_auc(model, val_loader)

        # --- decide whether to activate SWA (if not already active) ---
        if not swa_active:
            reason = None
            if epoch >= swa_target_start:
                reason = "scheduled"
            elif patience_ctr >= patience:
                reason = "patience_forced"
            if reason is not None:
                swa_active           = True
                swa_activation_epoch = epoch
                pre_swa_best_auc     = best_auc
                print(f"    [seed {seed}] SWA activated at P2 ep {epoch} "
                      f"({reason}, lr→{swa_lr:.6f}, budget={swa_budget} ep)")
                print(f"    [seed {seed}]   ckpt frozen at AUC={pre_swa_best_auc:.4f}")

        # --- step optimizer / SWA state ---
        if swa_active:
            swa_sched.step()
            swa_model.update_parameters(model)
        else:
            sched2.step()

        history['loss'].append(loss); history['val_auc'].append(val_auc); history['phase'].append(2)
        lr_now  = opt2.param_groups[1]['lr']

        # --- ckpt + patience: only when SWA is NOT active ---
        if not swa_active:
            is_best = val_auc > best_auc
            if is_best:
                best_auc = val_auc; patience_ctr = 0
                torch.save(model.state_dict(), ckpt)
            else:
                patience_ctr += 1
            tag = "←" if is_best else f" (p{patience_ctr}/{patience})"
            swa_tag = ""
        else:
            # During SWA, validate but DO NOT touch ckpt or patience.
            # Print SWA-internal AUC for monitoring only.
            tag = ""
            swa_tag = "[SWA]"

        actual_p2_epochs = epoch
        print(f"    P2 {epoch:02d} {swa_tag} | LR:{lr_now:.6f} | "
              f"Loss:{loss:.4f} | AUC:{val_auc:.4f}  {tag}")

        # --- SWA budget exhausted? ---
        if swa_active and (epoch - swa_activation_epoch + 1) >= swa_budget:
            print(f"    [seed {seed}] SWA budget complete at P2 ep {epoch} "
                  f"({swa_budget} ep run). Finalizing.")
            break

    # SWA: finalize by recomputing BN running stats on train data
    if swa_active:
        print(f"    [seed {seed}] Finalizing SWA: recomputing BN stats...")
        # update_bn iterates the loader; for tuple batches it uses input[0].
        # Our collate yields (imgs, labels, orig_imgs, paths) — imgs is element 0,
        # so update_bn picks the right thing without a wrapper.
        update_bn(train_loader, swa_model, device=CFG.DEVICE)
        # Evaluate SWA model — keep it ONLY if it beats the best-AUC checkpoint.
        # This avoids regressing in cases where SWA averaged through a bad region.
        swa_auc = evaluate_auc(swa_model, val_loader)
        print(f"    [seed {seed}] SWA AUC={swa_auc:.4f}  vs  best ckpt AUC={best_auc:.4f}")
        if swa_auc > best_auc:
            print(f"    [seed {seed}] SWA wins — using SWA weights.")
            chosen = swa_model
        else:
            print(f"    [seed {seed}] Best ckpt wins — using ckpt weights.")
            model.load_state_dict(torch.load(ckpt, map_location=CFG.DEVICE))
            chosen = model
    else:
        # We finished unfreeze_epochs without SWA ever activating.
        # That should be impossible now with force-on-patience, but kept for safety.
        print(f"    [seed {seed}] Note: SWA never activated (ran to end of P2). Using ckpt.")
        model.load_state_dict(torch.load(ckpt, map_location=CFG.DEVICE))
        chosen = model

    # Temperature scaling (multi-step LBFGS — v0-10 fix #1)
    scaled = TemperatureScaler(chosen).to(CFG.DEVICE)
    T = scaled.calibrate(val_loader)
    print(f"    [seed {seed}] Temperature T={T:.3f}")

    return scaled, history, T

# ╔══════════════════════════════════════════════════════════════╗
# ║  8. TRAIN ALL SEEDS + ENSEMBLE FOR ONE MAGNIFICATION       ║
# ╚══════════════════════════════════════════════════════════════╝
def train_and_evaluate(mag):
    mc            = MAG_CFG[mag]
    img_size      = mc['IMG_SIZE']
    aug_strength  = mc['AUG_STRENGTH']
    n_seeds_mag   = mc['N_SEEDS']

    print(f"\n{'#'*62}")
    print(f"  MAGNIFICATION: {mag}")
    print(f"  backbone=resnet50  img_size={img_size}  aug={aug_strength}")
    print(f"  freeze={mc['FREEZE_EPOCHS']}  unfreeze={mc['UNFREEZE_EPOCHS']}  "
          f"dropout={mc['DROPOUT_1']}  patience={mc['PATIENCE']}  wd={mc['WEIGHT_DECAY']:.0e}")
    print(f"  lr_p1={mc['LR_P1']:.0e}  lr_head={mc['LR_HEAD_P2']:.0e}  "
          f"lr_bb={mc['LR_BACKBONE_P2']:.0e}  p2_warmup={mc['P2_WARMUP']}")
    print(f"  N_SEEDS={n_seeds_mag}  SWA_FRAC={CFG.SWA_FRAC}  "
          f"TTA_5crop={CFG.TTA_FIVE_CROP}  AMP={CFG.AMP_ENABLED}")
    print(f"{'#'*62}")

    t_train  = get_train_transform(img_size, strength=aug_strength)
    t_val    = get_val_transform(img_size)
    # Build datasets once (v0-8 was building val_ds twice — same files, double the os.walk)
    train_ds = HistologyDataset(mag, 'train', t_train)
    val_ds   = HistologyDataset(mag, 'test',  t_val)
    train_loader = make_train_loader(train_ds)
    val_loader   = make_val_loader(val_ds)
    print(f"  Train={len(train_ds)}  Test={len(val_ds)}\n")

    # ── Train N seeds, collect calibrated TTA probabilities ──
    seed_probs_list = []
    seed_results    = []
    last_model      = None
    last_history    = None
    y_true_ref      = None
    orig_imgs_ref   = None
    seeds_to_run    = CFG.SEEDS[:n_seeds_mag]

    for seed in seeds_to_run:
        print(f"\n  ━━━ {mag}  seed={seed}  ({seeds_to_run.index(seed)+1}/{len(seeds_to_run)}) ━━━")
        t0 = time.time()
        scaled, history, T = train_one_seed(mag, seed, mc, train_loader, val_loader)
        y_prob, y_true, orig_imgs = tta_predict(scaled, val_loader, img_size)
        seed_auc = roc_auc_score(y_true, y_prob)
        print(f"  [seed {seed}] standalone TTA AUC = {seed_auc:.4f}   ({(time.time()-t0)/60:.1f} min)")
        seed_probs_list.append(y_prob)
        seed_results.append({'seed': seed, 'auc': seed_auc, 'T': T})
        # Hold on to one model for XAI (use the first seed's calibrated model)
        if last_model is None:
            last_model    = scaled.model      # underlying Res_MSHRA (or AveragedModel wrapper)
            last_history  = history
            y_true_ref    = y_true
            orig_imgs_ref = orig_imgs

    # ── Ensemble: average TTA probabilities across seeds ──
    y_prob = np.mean(np.stack(seed_probs_list, axis=0), axis=0)
    y_true = y_true_ref
    roc_auc_val = roc_auc_score(y_true, y_prob)
    seed_auc_str = ", ".join("{:.4f}".format(r['auc']) for r in seed_results)
    print(f"\n  ═══ {mag} ENSEMBLE AUC = {roc_auc_val:.4f}  "
          f"(seeds: [{seed_auc_str}]) ═══")

    best_thr, best_acc = accuracy_optimal_threshold(y_true, y_prob)
    y_pred = (y_prob >= best_thr).astype(int)

    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    thr_youden = thresholds[np.argmax(tpr - fpr)]
    acc_youden = accuracy_score(y_true, (y_prob >= thr_youden).astype(int))

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    ece = plot_calibration_for_mag(y_true, y_prob, mag)

    print(f"\n{'='*62}")
    print(f"  RESULTS: {mag}  (ensemble of {n_seeds_mag} seeds)")
    print(f"{'='*62}")
    print(f"  ROC-AUC     : {roc_auc_val:.4f}")
    print(f"  Accuracy    : {best_acc*100:.2f}%  (acc-optimal thr={best_thr:.4f})")
    print(f"  Sensitivity : {tp/(tp+fn):.4f}")
    print(f"  Specificity : {tn/(tn+fp):.4f}")
    print(f"  Precision   : {precision_score(y_true,y_pred):.4f}")
    print(f"  F1-score    : {f1_score(y_true,y_pred):.4f}")
    print(f"  ECE         : {ece:.4f}")
    print(f"  TP={tp}  TN={tn}  FP={fp}  FN={fn}")
    print(f"  Per-seed AUCs: [{seed_auc_str}]")
    print(f"  Per-seed std : {np.std([r['auc'] for r in seed_results]):.4f}")
    print(classification_report(y_true, y_pred, target_names=['Benign','Malignant']))

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    sns.heatmap(confusion_matrix(y_true,y_pred), annot=True, fmt='d', cmap='Purples',
                ax=axes[0], xticklabels=['Benign','Malignant'],
                yticklabels=['Benign','Malignant'])
    axes[0].set_title(f'{mag} confusion matrix')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
    axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC={roc_auc_val:.4f}')
    axes[1].fill_between(fpr, tpr, alpha=0.08, color='darkorange')
    axes[1].plot([0,1],[0,1],'k--',lw=1); axes[1].legend(loc='lower right')
    axes[1].set_title(f'{mag} ROC curve (ensemble)')
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[2].hist(y_prob[y_true==0],bins=25,alpha=0.65,label='Benign',color='steelblue')
    axes[2].hist(y_prob[y_true==1],bins=25,alpha=0.65,label='Malignant',color='tomato')
    axes[2].axvline(best_thr,color='black',linestyle='--',label=f'thr={best_thr:.2f}')
    axes[2].set_title(f'{mag} probability distribution'); axes[2].legend()
    plt.suptitle(f"Res-MSHRA v0-10  |  {mag}  |  AUC={roc_auc_val:.4f}  Acc={best_acc*100:.1f}%  "
                 f"({n_seeds_mag}-seed ensemble + SWA)", fontsize=13)
    plt.tight_layout(); plt.show()

    # Training history (from the first seed, which is the model used for XAI)
    phases = np.array(last_history['phase']); ep = np.arange(1, len(last_history['loss'])+1)
    p1_end = int(np.where(phases==1)[0][-1]) + 1
    fig, axes = plt.subplots(1,2,figsize=(13,4))
    axes[0].plot(ep, last_history['loss'], color='steelblue')
    axes[0].axvline(p1_end,color='gray',linestyle=':',alpha=0.7,label='P1→P2')
    axes[0].set_title(f'{mag} training loss (seed {seeds_to_run[0]})'); axes[0].legend()
    axes[1].plot(ep, last_history['val_auc'], color='darkorange')
    axes[1].axvline(p1_end,color='gray',linestyle=':',alpha=0.7,label='P1→P2')
    axes[1].axhline(roc_auc_val,linestyle='--',color='red',alpha=0.5,label=f'Ensemble AUC={roc_auc_val:.4f}')
    axes[1].set_title(f'{mag} val AUC (seed {seeds_to_run[0]})'); axes[1].legend()
    plt.tight_layout(); plt.show()

    # XAI uses the first-seed model. Note: if SWA won for seed-0, model is wrapped
    # in AveragedModel — its forward delegates to the underlying module, and the
    # mshra/sa hooks below still resolve correctly because AveragedModel.module
    # is the original Res_MSHRA. We pass that through.
    xai_model = last_model.module if isinstance(last_model, AveragedModel) else last_model
    print(f"\n  Running XAI for {mag} (using seed-{seeds_to_run[0]} model)...")
    plot_gradcam_for_mag(xai_model, val_ds, y_prob, mag, img_size)
    plot_attention_for_mag(xai_model, val_ds, y_prob, mag, img_size)
    plot_prediction_grid_for_mag(val_ds, y_prob, y_pred, mag)

    return {
        'mag': mag, 'auc': roc_auc_val, 'acc': best_acc,
        'sensitivity': tp/(tp+fn), 'specificity': tn/(tn+fp),
        'precision': precision_score(y_true,y_pred),
        'f1': f1_score(y_true,y_pred), 'ece': ece,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'threshold': best_thr,
        'fpr': fpr, 'tpr': tpr,
        'n_train': len(train_loader.dataset), 'n_test': len(val_ds),
        'per_seed_aucs': [r['auc'] for r in seed_results],
        'per_seed_std' : float(np.std([r['auc'] for r in seed_results])),
    }

# ╔══════════════════════════════════════════════════════════════╗
# ║  9. RUN ALL MAGNIFICATIONS                                 ║
# ╚══════════════════════════════════════════════════════════════╝
all_results = {}
overall_t0  = time.time()
for mag in CFG.MAGNIFICATIONS:
    result = train_and_evaluate(mag)
    all_results[mag] = result
    elapsed_min = (time.time() - overall_t0) / 60
    print(f"\n  ⏱  Total elapsed so far: {elapsed_min:.1f} min  "
          f"(Kaggle session limit ≈ 540 min for T4×2 / 720 min for older kernels)")

# ╔══════════════════════════════════════════════════════════════╗
# ║  10. CROSS-MAGNIFICATION COMPARISON                        ║
# ╚══════════════════════════════════════════════════════════════╝
print(f"\n\n{'#'*62}")
print("  CROSS-MAGNIFICATION COMPARISON  (v0-10, ensemble + SWA)")
print(f"{'#'*62}")

colors = {'40X':'#e74c3c', '100X':'#f39c12', '200X':'#2ecc71', '400X':'#3498db'}
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for mag, res in all_results.items():
    axes[0].plot(res['fpr'], res['tpr'], color=colors[mag], lw=2.5,
                 label=f"{mag}  AUC={res['auc']:.4f}")
axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set_title('ROC Curves — All Magnifications', fontsize=13)
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate', fontsize=11)
axes[0].legend(fontsize=10, loc='lower right')
axes[0].grid(alpha=0.3)

mags     = list(all_results.keys())
metrics  = ['auc', 'acc', 'sensitivity', 'specificity', 'f1']
x        = np.arange(len(mags))
width    = 0.15

for i, metric in enumerate(metrics):
    vals = [all_results[m][metric] for m in mags]
    axes[1].bar(x + i*width, vals, width, label=metric.capitalize(), alpha=0.85)

axes[1].set_xticks(x + width*2)
axes[1].set_xticklabels(mags)
axes[1].set_ylim(0.7, 1.02)
axes[1].set_title('Metrics by Magnification', fontsize=13)
axes[1].set_ylabel('Score', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3, axis='y')

plt.suptitle('Res-MSHRA v0-10 — Cross-Magnification Analysis', fontsize=14)
plt.tight_layout(); plt.show()

print(f"\n{'='*108}")
print("  PAPER-READY RESULTS TABLE  (v0-10)")
print(f"{'='*108}")
print(f"  {'Mag':<6} {'#S':>3} {'Train':>7} {'Test':>6} {'AUC':>7} {'Acc%':>7} {'Sens':>7} "
      f"{'Spec':>7} {'Prec':>7} {'F1':>7} {'ECE':>7} {'σ(seeds)':>9}")
print(f"  {'-'*106}")
for mag, r in all_results.items():
    n_s = MAG_CFG[mag]['N_SEEDS']
    print(f"  {mag:<6} {n_s:>3} {r['n_train']:>7} {r['n_test']:>6} {r['auc']:>7.4f} "
          f"{r['acc']*100:>6.2f}% {r['sensitivity']:>7.4f} {r['specificity']:>7.4f} "
          f"{r['precision']:>7.4f} {r['f1']:>7.4f} {r['ece']:>7.4f} {r['per_seed_std']:>9.4f}")
print(f"  {'-'*106}")

avg_auc  = np.mean([r['auc'] for r in all_results.values()])
avg_acc  = np.mean([r['acc'] for r in all_results.values()])
avg_sens = np.mean([r['sensitivity'] for r in all_results.values()])
avg_spec = np.mean([r['specificity'] for r in all_results.values()])
avg_prec = np.mean([r['precision'] for r in all_results.values()])
avg_f1   = np.mean([r['f1'] for r in all_results.values()])
avg_ece  = np.mean([r['ece'] for r in all_results.values()])
print(f"  {'Average':<6} {'':>3} {'':>7} {'':>6} {avg_auc:>7.4f} {avg_acc*100:>6.2f}% "
      f"{avg_sens:>7.4f} {avg_spec:>7.4f} {avg_prec:>7.4f} {avg_f1:>7.4f} {avg_ece:>7.4f}")
print(f"{'='*108}")

print(f"\n  Best single magnification: "
      f"{max(all_results, key=lambda m: all_results[m]['auc'])}  "
      f"(AUC={max(r['auc'] for r in all_results.values()):.4f})")
print(f"  Best accuracy: "
      f"{max(all_results, key=lambda m: all_results[m]['acc'])}  "
      f"({max(r['acc'] for r in all_results.values())*100:.2f}%)")

# Per-seed breakdown (useful when writing up: shows ensemble lift)
print(f"\n  Per-seed AUC breakdown:")
for mag, r in all_results.items():
    seed_aucs = r['per_seed_aucs']
    seeds_fmt = ", ".join("{:.4f}".format(a) for a in seed_aucs)
    print(f"    {mag}: seeds=[{seeds_fmt}]  "
          f"mean={np.mean(seed_aucs):.4f}  ensemble={r['auc']:.4f}  "
          f"lift=+{(r['auc']-np.mean(seed_aucs))*100:.2f}pp")

metrics_data = np.array([
    [all_results[m]['auc'], all_results[m]['acc'],
     all_results[m]['sensitivity'], all_results[m]['specificity'],
     all_results[m]['f1'], 1-all_results[m]['ece']]
    for m in CFG.MAGNIFICATIONS
])
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(metrics_data, annot=True, fmt='.4f', cmap='YlOrRd',
            xticklabels=['AUC','Accuracy','Sensitivity','Specificity','F1','1-ECE'],
            yticklabels=CFG.MAGNIFICATIONS, vmin=0.85, vmax=1.0,
            linewidths=0.5, ax=ax)
ax.set_title('Res-MSHRA v0-10 Performance Heatmap — All Magnifications', fontsize=13)
plt.tight_layout(); plt.show()

print(f"\n✓ All magnification experiments complete in {(time.time()-overall_t0)/60:.1f} min.")
print(f"  Checkpoints saved in {CFG.OUT_DIR}/")
